In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import (
    train_test_split,
    cross_validate,
    KFold,
    GridSearchCV
)

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from sklearn.ensemble import (
    GradientBoostingRegressor,
    RandomForestRegressor
)

from sklearn.linear_model import Ridge, Lasso, RidgeCV
from sklearn.neighbors import KNeighborsRegressor
import math
import matplotlib.pyplot as plt
import seaborn as sns
import joblib


In [ ]:
# load from joblib

df_encoded = joblib.load('df_encoded.joblib')
df_encoded

In [ ]:
y = df_encoded["Price_log"] 
y

In [ ]:
X = df_encoded.drop(["Group_median", "Group_std", "Price", "Price_log"], axis=1)
print("""we are not using  column "Group_median", "Group_std", "Price" to train 
because they were gotten from  price directly and price has skewed data
""")
X

In [ ]:
import matplotlib.pyplot as plt
import math

# Feature names
feature_names = [
    'Bedrooms',
    'Is_premium_subarea',
    'Is_estate',
    'Price_zscore_in_group',
    'Locality_encoded',
    'Area_encoded',
    'Property_type_encoded'
]

# Number of features
n_features = X.shape[1]

# Grid layout
n_cols = 3
n_rows = math.ceil(n_features / n_cols)

# Create subplots
fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(5 * n_cols, 4 * n_rows)
)

# Flatten axes
axes = axes.flatten()

# Plot each feature
for i in range(n_features):

    axes[i].scatter(X.iloc[:, i], y)

    axes[i].set_xlabel(feature_names[i])
    axes[i].set_ylabel("y")

    axes[i].set_title(f"{feature_names[i]} vs y")

# Remove unused empty plots
for j in range(n_features, len(axes)):
    fig.delaxes(axes[j])

# Improve spacing
plt.tight_layout()

# Show plots
plt.show()

In [ ]:
# Train/Test Split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
# Gradient Boosting Model
gbr = GradientBoostingRegressor(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=4,
    min_samples_split=5,
    min_samples_leaf=3,
    subsample=0.8,
    random_state=42
)

# KFold Cross Validation
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# cross_validate()
cv_results = cross_validate(
    estimator=gbr,
    X=X_train,
    y=y_train,
    cv=kf,
    scoring=(
        'neg_root_mean_squared_error',
        'neg_mean_absolute_error',
        'r2'
    ),
    return_train_score=True,
    n_jobs=-1
)
print(cv_results.keys())

# RMSE
cv_rmse = -cv_results['test_neg_root_mean_squared_error']
print("Fold RMSE:", cv_rmse)
print("Mean RMSE:", cv_rmse.mean())

# MAE
cv_mae = -cv_results['test_neg_mean_absolute_error']
print("Fold MAE:", cv_mae)
print("Mean MAE:", cv_mae.mean())

# R²
cv_r2 = cv_results['test_r2']
print("Fold R2:", cv_r2)
print("Mean R2:", cv_r2.mean())


In [ ]:
# normal testing
gbr.fit(X_train, y_train)
log_preds = gbr.predict(X_test)
real_preds = np.expm1(log_preds)
real_y_test = np.expm1(y_test)

# Final Evaluation on Real Prices
rmse = np.sqrt(
    mean_squared_error(real_y_test, real_preds)
)

mae = mean_absolute_error(
    real_y_test,
    real_preds
)

r2 = r2_score(
    real_y_test,
    real_preds
)

print("Test RMSE:", rmse)
print("Test MAE:", mae)
print("Test R2:", r2)

In [ ]:
# Detect Overfitting
train_rmse = -cv_results['train_neg_root_mean_squared_error'].mean()

test_rmse = -cv_results['test_neg_root_mean_squared_error'].mean()

print(train_rmse)
print(test_rmse)
print('''If:

train RMSE is much lower than test RMSE
→ model is overfitting.''')

In [ ]:
# Cross validate for randomforest

# Random Forest Model
rf = RandomForestRegressor(
    n_estimators=400,
    max_depth=15,
    min_samples_split=4,
    min_samples_leaf=2,
    max_features='sqrt',
    bootstrap=True,
    random_state=42,
    n_jobs=-1
)

# KFold Setup
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# cross_validate()
cv_results = cross_validate(
    estimator=rf,
    X=X_train,
    y=y_train,
    cv=kf,
    scoring=(
        'neg_root_mean_squared_error',
        'neg_mean_absolute_error',
        'r2'
    ),
    return_train_score=True,
    n_jobs=-1
)
print(cv_results.keys())

# RMSE Scores
cv_rmse = -cv_results['test_neg_root_mean_squared_error']
print("Fold RMSE:", cv_rmse)
print("Mean RMSE:", cv_rmse.mean())

#MAE Scores
cv_mae = -cv_results['test_neg_mean_absolute_error']
print("Fold MAE:", cv_mae)
print("Mean MAE:", cv_mae.mean())

# R² Scores
cv_r2 = cv_results['test_r2']
print("Fold R2:", cv_r2)
print("Mean R2:", cv_r2.mean())


In [ ]:
# Detect Overfitting, Compare train vs validation RMSE.

train_rmse = -cv_results[
    'train_neg_root_mean_squared_error'
].mean()

test_rmse = -cv_results[
    'test_neg_root_mean_squared_error'
].mean()

print("Train RMSE:", train_rmse)
print("Validation RMSE:", test_rmse)

In [ ]:
# Train final model
#rf.fit(X_train, y_train)

# Convert Back to Real Prices Since target is log-transformed:
#real_preds = np.expm1(log_preds)

# Convert Back to Real Prices because target is log-transformed:

#real_preds = np.expm1(log_preds)

#real_y_test = np.expm1(y_test)

In [ ]:
# KNN Pipeline
knn_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', KNeighborsRegressor(
        n_neighbors=8,
        weights='distance',
        p=2
    ))
])

# KFold Cross Validation
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# cross_validate()
cv_results = cross_validate(
    estimator=knn_pipeline,
    X=X_train,
    y=y_train,
    cv=kf,
    scoring=(
        'neg_root_mean_squared_error',
        'neg_mean_absolute_error',
        'r2'
    ),
    return_train_score=True,
    n_jobs=-1
)
print(cv_results.keys())

# RMSE
cv_rmse = -cv_results[
    'test_neg_root_mean_squared_error'
]
print("Fold RMSE:", cv_rmse)
print("Mean RMSE:", cv_rmse.mean())

# MAE
cv_mae = -cv_results[
    'test_neg_mean_absolute_error'
]
print("Fold MAE:", cv_mae)
print("Mean MAE:", cv_mae.mean())

# R²
cv_r2 = cv_results['test_r2']
print("Fold R2:", cv_r2)
print("Mean R2:", cv_r2.mean())


In [ ]:
# Check Overfitting
train_rmse = -cv_results[
    'train_neg_root_mean_squared_error'
].mean()

test_rmse = -cv_results[
    'test_neg_root_mean_squared_error'
].mean()

print("Train RMSE:", train_rmse)
print("Validation RMSE:", test_rmse)

In [ ]:
knn_pipeline.fit(X_train, y_train)
log_preds = knn_pipeline.predict(X_test)
real_preds = np.expm1(log_preds)
real_y_test = np.expm1(y_test)